# PCNA Apo MD Simulation — GNN-PCNA Project

Runs a 100 ns all-atom explicit-solvent MD simulation of apo PCNA (PDB 1W60) using OpenMM.

**Runtime required:** GPU (T4 or A100). Go to Runtime → Change runtime type → GPU.

**Outputs:**
- `1W60_production.xtc` — trajectory (10 ps/frame, 10,000 frames for 100 ns)
- `1W60_production.pdb` — topology for MDAnalysis
- `1W60_summary.json` — per-run metadata

**After this notebook:** download the outputs and run `python scripts/run_md_analysis.py` locally.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys

subprocess.run([
    'conda', 'install', '-y', '-c', 'conda-forge',
    'openmm', 'pdbfixer', 'mdanalysis', 'mdtraj'
], check=False)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'openmm', 'pdbfixer', 'MDAnalysis', 'mdtraj'
], check=False)

import openmm
print('OpenMM version:', openmm.__version__)

# Verify GPU
from openmm import Platform
platforms = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
print('Available platforms:', platforms)
if 'CUDA' in platforms:
    print('GPU (CUDA) available — fast path enabled')
elif 'OpenCL' in platforms:
    print('OpenCL available')
else:
    print('WARNING: Only CPU platform found — simulation will be very slow')

In [ ]:
# ── Cell 2: Upload 1W60.pdb ────────────────────────────────────────────────────
# Option A: upload from your machine
from google.colab import files
import os

print('Upload 1W60.pdb from data/raw/1W60.pdb in the GNN_PCNA repo')
uploaded = files.upload()

# Rename to standard name if needed
for fn in uploaded:
    if fn != '1W60.pdb':
        os.rename(fn, '1W60.pdb')
    print(f'Loaded: {fn} ({len(uploaded[fn])} bytes)')

# Option B (uncomment to use RCSB directly instead of uploading)
# import urllib.request
# urllib.request.urlretrieve('https://files.rcsb.org/download/1W60.pdb', '1W60.pdb')
# print('Downloaded 1W60.pdb from RCSB')

In [ ]:
# ── Cell 3: Fix structure with PDBFixer ────────────────────────────────────────
# Adds missing residues, missing heavy atoms, and hydrogens.
# Removes HETATM (no ligand in 1W60 apo), keeps chains A and B.
from pdbfixer import PDBFixer
from openmm.app import PDBFile

fixer = PDBFixer(filename='1W60.pdb')

print('Chains in file:', [str(c.id) for c in fixer.topology.chains()])
print('Finding missing residues...')
fixer.findMissingResidues()
print(f'Missing residues: {len(fixer.missingResidues)}')

print('Finding missing atoms...')
fixer.findMissingAtoms()
print(f'Missing atoms: {sum(len(v) for v in fixer.missingAtoms.values())}')

print('Adding missing atoms...')
fixer.addMissingAtoms()

print('Adding hydrogens (pH 7.4)...')
fixer.addMissingHydrogens(7.4)

# Save fixed structure
with open('1W60_fixed.pdb', 'w') as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)

n_atoms = fixer.topology.getNumAtoms()
n_res   = fixer.topology.getNumResidues()
print(f'\nFixed structure: {n_atoms} atoms, {n_res} residues')
print('Saved: 1W60_fixed.pdb')

In [ ]:
# ── Cell 4: Build simulation system ────────────────────────────────────────────
# Force field: CHARMM36m (protein) + TIP3P-FB (water)
# Box: dodecahedral, 12 Å padding, 150 mM NaCl
from openmm.app import (
    ForceField, Modeller, PME, HBonds, PDBFile
)
from openmm import unit

print('Loading force field (CHARMM36m + tip3p)...')
forcefield = ForceField('charmm36.xml', 'charmm36/water.xml')

pdb = PDBFile('1W60_fixed.pdb')
modeller = Modeller(pdb.topology, pdb.positions)

# Solvate: dodecahedral box, 12 Å padding, 150 mM NaCl
print('Solvating (12 Å padding, 150 mM NaCl)...')
modeller.addSolvent(
    forcefield,
    model='tip3p',
    padding=12 * unit.angstrom,
    ionicStrength=0.15 * unit.molar,
    positiveIon='Na+',
    negativeIon='Cl-',
)
n_atoms_solv = modeller.topology.getNumAtoms()
print(f'Solvated system: {n_atoms_solv} atoms')

# Build OpenMM system
print('Creating OpenMM system...')
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0 * unit.nanometer,
    constraints=HBonds,
    hydrogenMass=4 * unit.amu,   # HMR allows 4 fs timestep
)

# Save topology PDB for MDAnalysis
with open('1W60_topology.pdb', 'w') as f:
    PDBFile.writeFile(modeller.topology, modeller.positions, f)
print('Saved topology: 1W60_topology.pdb')

In [ ]:
# ── Cell 5: Energy minimization ─────────────────────────────────────────────────
from openmm import LangevinMiddleIntegrator, MonteCarloBarostat
from openmm.app import Simulation, StateDataReporter
from openmm import unit
import sys

# Pick best platform
from openmm import Platform
platform_name = 'CUDA' if 'CUDA' in [Platform.getPlatform(i).getName()
                                       for i in range(Platform.getNumPlatforms())] else 'CPU'
platform = Platform.getPlatformByName(platform_name)
if platform_name == 'CUDA':
    properties = {'CudaPrecision': 'mixed'}
else:
    properties = {}

print(f'Using platform: {platform_name}')

integrator = LangevinMiddleIntegrator(
    310 * unit.kelvin,    # physiological temperature
    1.0 / unit.picosecond,
    4.0 * unit.femtosecond,  # 4 fs with HMR
)

simulation = Simulation(
    modeller.topology, system, integrator, platform, properties
)
simulation.context.setPositions(modeller.positions)

print('Energy minimizing...')
simulation.minimizeEnergy(maxIterations=2000)
state = simulation.context.getState(getEnergy=True)
print(f'Minimized PE: {state.getPotentialEnergy()}')

In [ ]:
# ── Cell 6: NVT equilibration (1 ns, 310 K) ────────────────────────────────────
from openmm.app import StateDataReporter
import sys

NVT_STEPS = 250_000   # 250,000 × 4 fs = 1 ns

simulation.reporters.clear()
simulation.reporters.append(
    StateDataReporter(sys.stdout, 25_000,
        step=True, potentialEnergy=True, temperature=True, speed=True)
)

# Gradually heat from 100 K to 310 K over 500 ps
from openmm import unit
integrator.setTemperature(100 * unit.kelvin)
for T in range(100, 311, 10):
    integrator.setTemperature(T * unit.kelvin)
    simulation.step(5_000)   # 20 ps per temperature step

print('\nNVT equilibration at 310 K...')
simulation.step(NVT_STEPS - 105_000)  # remaining NVT
print('NVT done')

In [ ]:
# ── Cell 7: NPT equilibration (1 ns, 310 K, 1 atm) ────────────────────────────
from openmm import MonteCarloBarostat
from openmm import unit

NPT_STEPS = 250_000   # 1 ns

# Add barostat for NPT
barostat = MonteCarloBarostat(1.0 * unit.bar, 310 * unit.kelvin, 25)
system.addForce(barostat)
simulation.context.reinitialize(preserveState=True)

print('NPT equilibration (310 K, 1 atm)...')
simulation.reporters.clear()
simulation.reporters.append(
    StateDataReporter(sys.stdout, 25_000,
        step=True, potentialEnergy=True, temperature=True,
        volume=True, density=True, speed=True)
)
simulation.step(NPT_STEPS)
print('NPT done')

# Save equilibrated state
state = simulation.context.getState(getPositions=True, getVelocities=True)
with open('1W60_equil.xml', 'w') as f:
    f.write(simulation.context.createCheckpoint().decode('latin-1'))
simulation.saveCheckpoint('1W60_equil.chk')
print('Checkpoint saved: 1W60_equil.chk')

In [ ]:
# ── Cell 8: Production run (100 ns) ────────────────────────────────────────────
# Writes XTC trajectory every 10 ps (4 fs × 2500 steps = 10 ps)
# 100 ns / 10 ps = 10,000 frames  → XTC file ~5-10 GB
from openmm.app import DCDReporter, StateDataReporter, CheckpointReporter
import sys, time

PROD_STEPS  = 25_000_000   # 25M × 4 fs = 100 ns
SAVE_EVERY  = 2_500        # 2,500 × 4 fs = 10 ps per frame
LOG_EVERY   = 250_000      # log every 1 ns

# Use DCD (MDAnalysis reads DCD natively; XTC needs mdtraj conversion)
simulation.reporters.clear()

simulation.reporters.append(
    DCDReporter('1W60_production.dcd', SAVE_EVERY)
)
simulation.reporters.append(
    StateDataReporter('1W60_production.log', LOG_EVERY,
        step=True, time=True, potentialEnergy=True,
        kineticEnergy=True, temperature=True, volume=True,
        speed=True, progress=True, remainingTime=True,
        totalSteps=PROD_STEPS)
)
simulation.reporters.append(
    StateDataReporter(sys.stdout, LOG_EVERY,
        step=True, time=True, temperature=True, speed=True,
        progress=True, remainingTime=True, totalSteps=PROD_STEPS)
)
simulation.reporters.append(
    CheckpointReporter('1W60_production.chk', 2_500_000)  # checkpoint every 10 ns
)

print(f'Starting 100 ns production run ({PROD_STEPS:,} steps, saving every {SAVE_EVERY} steps)')
t0 = time.time()
simulation.step(PROD_STEPS)
elapsed = time.time() - t0
print(f'\nProduction complete in {elapsed/3600:.1f} hours')
print('Output: 1W60_production.dcd')

In [ ]:
# ── Cell 9: Quick MDAnalysis sanity check ──────────────────────────────────────
# Run directly in Colab to verify trajectory before downloading
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align
import numpy as np

u = mda.Universe('1W60_topology.pdb', '1W60_production.dcd')
print(f'Trajectory: {len(u.trajectory)} frames, {u.trajectory.dt} ps/frame')
print(f'Atoms: {u.atoms.n_atoms}, Residues: {u.residues.n_residues}')

# Align to first frame on backbone
ref = mda.Universe('1W60_topology.pdb')
aligner = align.AlignTraj(u, ref, select='backbone', in_memory=False).run()

# Per-residue Cα RMSF
ca = u.select_atoms('name CA')
rmsf_calc = rms.RMSF(ca).run()
rmsf = rmsf_calc.rmsf

print(f'\nCα RMSF summary:')
print(f'  Mean:   {rmsf.mean():.3f} Å')
print(f'  Median: {np.median(rmsf):.3f} Å')
print(f'  Max:    {rmsf.max():.3f} Å  (residue {ca.resids[rmsf.argmax()]})')

# AOH1996 pocket residues in 1W60
AOH_RESIDS = {25,26,27,38,39,40,41,42,44,45,46,47,
              123,125,126,128,231,232,233,234,250,251,252,253}
pocket_mask = np.array([r in AOH_RESIDS for r in ca.resids])
pocket_rmsf = rmsf[pocket_mask].mean()
bg_rmsf     = rmsf[~pocket_mask].mean()
print(f'\nAOH1996 pocket residues ({pocket_mask.sum()} matched):')
print(f'  Pocket RMSF: {pocket_rmsf:.3f} Å')
print(f'  Background:  {bg_rmsf:.3f} Å')
print(f'  Fold-change: {pocket_rmsf/bg_rmsf:.3f}')

# Save RMSF to CSV for GNN-PCNA analysis
import pandas as pd
df = pd.DataFrame({'resid': ca.resids, 'chain': ca.segids, 'rmsf_angstrom': rmsf})
df.to_csv('1W60_rmsf.csv', index=False)
print('\nSaved: 1W60_rmsf.csv')

In [ ]:
# ── Cell 10: Download results ───────────────────────────────────────────────────
# Download files needed for local GNN-PCNA analysis.
# 1W60_production.dcd is large (~5-10 GB) — use Google Drive for that.
import os
from google.colab import files, drive

# Mount Google Drive and copy trajectory there (avoids 12h Colab timeout losing data)
print('Mounting Google Drive to save large trajectory...')
drive.mount('/content/drive')
outdir = '/content/drive/MyDrive/GNN_PCNA_MD'
os.makedirs(outdir, exist_ok=True)

import shutil
for fname in ['1W60_production.dcd', '1W60_topology.pdb',
              '1W60_production.log', '1W60_equil.chk']:
    if os.path.exists(fname):
        shutil.copy(fname, outdir)
        size_mb = os.path.getsize(fname) / 1e6
        print(f'Copied {fname} ({size_mb:.0f} MB) -> Google Drive')

# Download small files directly
for fname in ['1W60_rmsf.csv', '1W60_production.log']:
    if os.path.exists(fname):
        files.download(fname)
        print(f'Downloaded: {fname}')

print('\nNext step:')
print('  Copy 1W60_production.dcd + 1W60_topology.pdb to GNN_PCNA/data/md/')
print('  Then run: python scripts/run_md_analysis.py')

## Expected timing on Colab T4 GPU

| Phase | Steps | Wall time |
|---|---|---|
| Energy minimization | 2,000 iter | ~2 min |
| NVT equilibration | 250,000 | ~15 min |
| NPT equilibration | 250,000 | ~15 min |
| Production (100 ns) | 25,000,000 | ~6–10 hours |

**Total: ~7–11 hours on T4. Use Colab Pro+ to avoid session timeout.**

For a faster test run (10 ns), change `PROD_STEPS = 2_500_000` in Cell 8.

## Output files for GNN-PCNA

| File | Size | Use |
|---|---|---|
| `1W60_production.dcd` | ~8 GB | Trajectory for MDAnalysis |
| `1W60_topology.pdb` | ~5 MB | Topology for MDAnalysis |
| `1W60_rmsf.csv` | ~10 KB | Per-residue RMSF (already computed in Cell 9) |
| `1W60_production.log` | ~50 KB | Energy/temperature log |

Place `1W60_production.dcd` and `1W60_topology.pdb` in `data/md/` then run:
```bash
python scripts/run_md_analysis.py
```